In [5]:
"""
PIPELINE STAGE: Data Extraction and Predictive Calculation (Subscriber Metrics)
PERIOD: January 2020 - October 2021
LIBRARIES: os, pandas, lxml, zipfile, re

1. OBJECTIVE:
   Scan annual directories under 'raw_data/energy_reports' to identify specific Word (.docx) tables. 
   Extract total subscriber counts by province and consumer type percentages. Calculate estimated 
   province-based subscriber counts by applying type percentages to total provincial figures.
   (Note: Billed electricity consumption extraction is mapped but currently pending implementation).

2. DIRECTORY STRUCTURE:
   - Source: os.path.join("..", "..", "raw_data", "energy_reports")
   - Output: os.path.join("..", "..", "processed_data", "steps", "01_2020_2021_subscriber.xlsx")

3. TECHNICAL WORKFLOW:
   A. XML Parsing: Open .docx files as zip archives and parse 'word/document.xml' directly for 
      high-precision table detection (bypassing standard docx library limitations).
   B. Header Reconstruction: Merge up to 5 preceding paragraphs ('w:p') to identify the table context accurately.
   C. Numerical Normalization (Turkish to Global Format):
      - Remove non-breaking spaces (\xa0), standard spaces, and thousand separators (dots).
      - Convert decimal commas to dots (e.g., 82,236 -> 82.236).
      - Handle null/error values as 0.0.

4. EXTRACTION & CALCULATION RULES:
   A. Consumer Type Percentages:
      - Target Header: "Tüketici Sayısının" AND "Tüketici Türü Bazında".
      - Categories: Mesken, Sanayi, Ticarethane, Tarımsal Sulama, Aydınlatma.
      - Logic: Extract 'Share (%)' from the 5th column (Index 4) and divide by 100 to get a decimal ratio (e.g., 82,236 -> 0.82236).
    
   B. Provincial Subscriber Totals:
      - Target Header: "Tüketici Sayısının" AND "İl Bazında".
      - Logic: Map dual-column layout (Left side: Col 0-1 | Right side: Col 4-5). Filter out 'TOPLAM' 
        and 'İl'/'il' header rows.
    
   C. Predictive Calculation (Estimated Subscriber Count):
      - For each province: [Category_Count] = [Total_Subscribers] * [Category_Percentage_Ratio].
      - Standardize category columns during mapping (Aydinlatma, Mesken, Sanayi, Tarim, Ticarethane).

5. OUTPUT:
   - Generate structured Excel files with consolidated subscriber metrics.
   - Inject 'Ay' (Month name/number) and 'Yil' (Year) dimensions for time-series tracking.
"""

import os
import re
import zipfile
import pandas as pd
from lxml import etree

# --- AYARLAR ---
BASE_PATH = os.path.join("..", "..", "raw_data", "energy_reports")
OUTPUT_TUKETICI = os.path.join("..", "..", "processed_data", "steps", "01_2020_2021_subscriber.xlsx")

GEÇERLİ_YILLAR = {
    "2020": list(range(1, 13)),
    "2021": list(range(1, 11))
}

def temizle_sayi(deger):
    if pd.isna(deger) or str(deger).strip() == "":
        return 0.0
    try:
        # TR formatını temizle: Noktayı kaldır, virgülü noktaya çevir
        s = str(deger).replace('\xa0', '').replace(' ', '').replace('.', '').replace(',', '.')
        return float(s)
    except:
        return 0.0

def ay_bul(dosya_adi):
    aylar = {"january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
 "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12}
    lower_name = dosya_adi.lower()
    for ay_isim, numara in aylar.items():
        if ay_isim in lower_name:
            return numara
    return 0

def tablo_ust_metin_birlestir(tbl_element, ns):
    parcalar = []
    current = tbl_element
    for _ in range(5):
        prev = current.getprevious()
        if prev is not None and prev.tag.endswith('p'):
            text = "".join(prev.xpath('.//w:t/text()', namespaces=ns)).strip()
            if text: parcalar.insert(0, text)
            current = prev
        else: break
    return re.sub(r'\s+', ' ', " ".join(parcalar))

def docx_analiz(yol):
    sonuclar = []
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
    try:
        with zipfile.ZipFile(yol) as docx:
            xml_content = docx.read('word/document.xml')
            tree = etree.fromstring(xml_content)
            for tbl in tree.xpath('//w:tbl', namespaces=ns):
                rows = []
                for tr in tbl.xpath('./w:tr', namespaces=ns):
                    cells = ["".join(tc.xpath('.//w:t/text()', namespaces=ns)).strip() for tc in tr.xpath('./w:tc', namespaces=ns)]
                    rows.append(cells)
                baslik = tablo_ust_metin_birlestir(tbl, ns)
                sonuclar.append({"baslik": baslik, "tablo": rows})
    except: return []
    return sonuclar

def verileri_isle():
    tuketici_birlesik_liste = []
    tuketim_birlesik_liste = []

    for yil in sorted(GEÇERLİ_YILLAR.keys()):
        gecerli_aylar = GEÇERLİ_YILLAR[yil]
        klasor_yolu = os.path.join(BASE_PATH, yil)
        if not os.path.exists(klasor_yolu): continue

        for dosya in os.listdir(klasor_yolu):
            if not dosya.endswith(".docx") or dosya.startswith("~$"): continue
            if ay_bul(dosya) not in gecerli_aylar: continue
            
            dosya_yolu = os.path.join(klasor_yolu, dosya)
            print(f"🔄 İşleniyor: {yil} / {dosya}")
            ay_adi = os.path.splitext(dosya)[0]

            try:
                analiz_listesi = docx_analiz(dosya_yolu)
                tuketici_tur_paylari = {}
                il_bazli_abone = []
                il_bazli_tuketim = []

                for öğe in analiz_listesi:
                    baslik = öğe["baslik"]
                    tablo = öğe["tablo"]

                    # --- 1. TABLO: ABONE PAYLARI (KARŞILAŞTIRMALI TABLO) ---
                    # Senin belirttiğin "5. sütundaki (indeks 4) Pay(%)" değerini alıyoruz
                    if "Tüketici Sayısının" in baslik and "Tüketici Türü Bazında" in baslik:
                        for satir in tablo:
                            if len(satir) >= 5: # En az 5 sütun olmalı
                                tur_adi = satir[0].strip()
                                if tur_adi in ["Aydınlatma", "Mesken", "Sanayi", "Tarımsal Sulama", "Ticarethane"]:
                                    # 5. sütun (indeks 4) pay değeridir. Örn: 82,236
                                    pay_degeri = temizle_sayi(satir[4]) 
                                    tuketici_tur_paylari[tur_adi] = pay_degeri / 100.0 # Yüzdeye çevir (0.82236)

                    # --- 2. TABLO: İL BAZLI ABONE SAYISI ---
                    if "Tüketici Sayısının" in baslik and "İl Bazında" in baslik:
                        for satir in tablo:
                            if len(satir) >= 2:
                                il_sol = satir[0].strip()
                                if il_sol and "TOPLAM" not in il_sol.upper() and il_sol != "il":
                                    il_bazli_abone.append({"il": il_sol, "Toplam_Abone": temizle_sayi(satir[1])})
                            if len(satir) >= 6:
                                il_sag = satir[4].strip()
                                if il_sag and "TOPLAM" not in il_sag.upper() and il_sag != "İl":
                                    il_bazli_abone.append({"il": il_sag, "Toplam_Abone": temizle_sayi(satir[5])})

                    # --- 3. TABLO: FATURALANAN TÜKETİM ---

                
                # --- HESAPLAMA ---
                if tuketici_tur_paylari and il_bazli_abone:
                    df_abone = pd.DataFrame(il_bazli_abone)
                    map_t = {'Aydinlatma':'Aydınlatma', 'Mesken':'Mesken', 'Sanayi':'Sanayi', 'Tarim':'Tarımsal Sulama', 'Ticarethane':'Ticarethane'}
                    for col, tur in map_t.items():
                        pay = tuketici_tur_paylari.get(tur, 0)
                        # Örn: 1.107.916 * 0.82236 = Doğru Mesken Abone Sayısı
                        df_abone[col] = df_abone['Toplam_Abone'] * pay
                    
                    df_abone['Ay'] = ay_adi.split('_')[0]
                    df_abone['Yil'] = yil
                    tuketici_birlesik_liste.append(df_abone)


                

                if il_bazli_tuketim:
                    tuketim_birlesik_liste.append(pd.DataFrame(il_bazli_tuketim))

            except Exception as e:
                print(f"❌ HATA: {dosya} -> {e}")

    # Kayıt
    if tuketici_birlesik_liste:
        pd.concat(tuketici_birlesik_liste, ignore_index=True).to_excel(OUTPUT_TUKETICI, index=False)
        print(f"✨ Abone Dosyası (5. Sütun Payı ile): {OUTPUT_TUKETICI}")

if __name__ == "__main__":
    verileri_isle()

🔄 İşleniyor: 2020 / April_2020.docx
🔄 İşleniyor: 2020 / August_2020.docx
🔄 İşleniyor: 2020 / December_2020.docx
🔄 İşleniyor: 2020 / February_2020.docx
🔄 İşleniyor: 2020 / January_2020.docx
🔄 İşleniyor: 2020 / July_2020.docx
🔄 İşleniyor: 2020 / June_2020.docx
🔄 İşleniyor: 2020 / March_2020.docx
🔄 İşleniyor: 2020 / May_2020.docx
🔄 İşleniyor: 2020 / November_2020.docx
🔄 İşleniyor: 2020 / October_2020.docx
🔄 İşleniyor: 2020 / September_2020.docx
🔄 İşleniyor: 2021 / April_2021.docx
🔄 İşleniyor: 2021 / August_2021.docx
🔄 İşleniyor: 2021 / February_2021.docx
🔄 İşleniyor: 2021 / January_2021.docx
🔄 İşleniyor: 2021 / July_2021.docx
🔄 İşleniyor: 2021 / June_2021.docx
🔄 İşleniyor: 2021 / March_2021.docx
🔄 İşleniyor: 2021 / May_2021.docx
🔄 İşleniyor: 2021 / October_2021.docx
🔄 İşleniyor: 2021 / September_2021.docx
✨ Abone Dosyası (5. Sütun Payı ile): ..\..\processed_data\steps\01_2020_2021_subscriber.xlsx
